<a href="https://colab.research.google.com/github/karthees512/The_Syntax_Surgeon.ipynb/blob/main/The_Syntax_Surgeon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gradio transformers torch black autopep8 reportlab requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.9/88.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.8/45.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 29.3 MB/s eta 0:00:00


In [4]:
import gradio as gr
from transformers import pipeline
import torch
import ast
import re
from datetime import datetime
from pathlib import Path
import black
import autopep8
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

# Configuration
class ChatbotConfig:
    MODEL_NAME = "ibm-granite/granite-3.3-2b-instruct"
    DEFAULT_TEMP = 0.7
    DEFAULT_MAX_TOKENS = 256
    DEFAULT_TOP_P = 0.95
    conversation_history = []
    model_pipe = None

# Initialize Model
def load_model():
    global model_pipe
    device = 0 if torch.cuda.is_available() else -1
    model_pipe = pipeline(
        "text-generation",
        model=ChatbotConfig.MODEL_NAME,
        device=device,
        torch_dtype=torch.float16 if device == 0 else torch.float32
    )
    return model_pipe

# Code Fixing Logic
def detect_python_code(text):
    patterns = [r'def\s+\w+', r'class\s+\w+', r'import\s+\w+', r'if\s+.*:', r'for\s+.*in']
    return any(re.search(p, text) for p in patterns)

def fix_python_code(code):
    # Indentation
    lines = code.split('\n')
    code = '\n'.join([line.rstrip().replace('\t', '    ') for line in lines])

    # Brackets
    for opening, closing in [('(', ')'), ('[', ']'), ('{', '}')]:
        open_count = code.count(opening) - code.count(closing)
        if open_count > 0:
            code += closing * open_count

    # Formatting
    try:
        return black.format_str(code, mode=black.FileMode())
    except:
        return autopep8.fix_code(code)

# Response Processing
def process_user_input(user_message, temp, tokens, top_p, chat_history):
    if detect_python_code(user_message):
        fixed = fix_python_code(user_message)
        response = f"Code fixed! Here is the corrected version:\n\n```python\n{fixed}\n```"
    else:
        # Standard AI response
        ChatbotConfig.conversation_history.append({"role": "user", "content": user_message})
        output = model_pipe(ChatbotConfig.conversation_history, max_new_tokens=int(tokens), temperature=temp, top_p=top_p, do_sample=True)
        response = output[0]['generated_text'][-1]['content']
        ChatbotConfig.conversation_history.append({"role": "assistant", "content": response})

    chat_history.append((user_message, response))
    return chat_history

# Gradio UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# The Syntax Surgeon")
    chatbot = gr.Chatbot(label="Chat History", height=400)
    msg = gr.Textbox(label="Message", lines=2)
    with gr.Row():
        temp_slider = gr.Slider(0.1, 1.0, value=0.7, label="Temp")
        token_slider = gr.Slider(50, 1024, value=256, label="Tokens")
    submit = gr.Button("Send", variant="primary")

    submit.click(process_user_input, [msg, temp_slider, token_slider, token_slider, chatbot], chatbot)

load_model()
demo.launch(share=True)

/tmp/ipykernel_794/244352108.py:73: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_794/244352108.py:75: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Chat History", height=400)
/tmp/ipykernel_794/244352108.py:75: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(label="Chat History", height=400)


Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c7f82748d6e17f4a3d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
